## **computer vision - lab 5 tasks**
### **image segmentation**
#### **23K-0074**

#### **setup**
images are in `images/` and every final result is saved to `output/`.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

os.makedirs("output", exist_ok=True)


def load(name, flag=cv2.IMREAD_COLOR):
    img = cv2.imread(os.path.join("images", name), flag)
    if img is None:
        raise FileNotFoundError(f"images/{name} not found")
    return img


def show(images, titles, cols=None, size=4, save=None):
    cols = cols or len(images)
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, im, t in zip(axes, images, titles):
        if im.ndim == 2:
            ax.imshow(im, cmap="gray", vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(t, fontsize=10)
    plt.tight_layout()
    if save:
        plt.savefig(os.path.join("output", save), dpi=110, bbox_inches="tight")
    plt.show()

### **task 1 - automated inspection of a document under uneven lighting**
image: `sudoku.png` (photographed newspaper sudoku; the lighting falls off towards the bottom/right).

In [ ]:
doc = load("sudoku.png", cv2.IMREAD_GRAYSCALE)
print("shape:", doc.shape, " mean intensity:", round(doc.mean(), 1))

# global thresholding with three values. THRESH_BINARY_INV so text/grid lines become white (foreground)
T_VALUES = [70, 100, 130]
global_masks = [cv2.threshold(doc, t, 255, cv2.THRESH_BINARY_INV)[1] for t in T_VALUES]

# adaptive threshold - method and parameters chosen in task 2 (mean, blockSize 51, C 15)
adaptive = cv2.adaptiveThreshold(doc, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 51, 15)

show([doc] + global_masks + [adaptive],
     ["original"] + [f"global T = {t}" for t in T_VALUES] + ["adaptive (mean, 51, C=15)"],
     cols=5, size=4, save="task1_comparison.png")
cv2.imwrite("output/task1_adaptive_mask.png", adaptive)

for t, m in zip(T_VALUES, global_masks):
    print(f"global T={t:3d}: {100 * (m > 0).mean():5.1f}% of pixels marked as ink")
print(f"adaptive     : {100 * (adaptive > 0).mean():5.1f}% of pixels marked as ink")

#### **where does global thresholding fail?**
the mean brightness of each quarter of the page shows the uneven lighting.

In [ ]:
h, w = doc.shape
quads = {"top-left": doc[:h//2, :w//2], "top-right": doc[:h//2, w//2:],
         "bottom-left": doc[h//2:, :w//2], "bottom-right": doc[h//2:, w//2:]}
for name, q in quads.items():
    print(f"{name:12s} mean brightness: {q.mean():6.1f}")

# background illumination estimate: a large blur removes the text and keeps only the lighting
illum = cv2.GaussianBlur(doc, (0, 0), 25)
show([doc, illum], ["original", "estimated illumination (heavy blur)"], size=5, save="task1_illumination.png")

**parameters chosen**
- **global T = 70 / 100 / 130:** spread around the image's mean brightness (101.8), so the test covers a threshold below, at and above the average.
- **adaptive:** `ADAPTIVE_THRESH_MEAN_C`, blockSize 51, C 15. this was the cleanest combination found in task 2.
- **`THRESH_BINARY_INV`:** used everywhere so that the ink (text and grid lines) is the white foreground.

**comparing the masks**
- **T = 70 (best global result):** the digits and grid in the top and right parts come out cleanly. the dark bottom-left corner of the page turns into a solid white blob, so digits there (e.g. the "3" and "2" near the bottom) are lost or merged with the background.
- **T = 100:** the white flood spreads across most of the left and bottom of the page. 45.3% of the pixels are marked as ink.
- **T = 130:** almost the whole page is white (77.8%); only the brightest top-right area still shows text.
- **adaptive:** every digit and grid line is readable across the whole page, including the dark corner. 15.8% of the pixels are marked as ink, which is close to the real amount of ink on the page. a little speckle remains around the folded page edges.

**which regions make global thresholding fail?**
- the quarter means show the lighting problem: top-right **126.6**, bottom-right 111.2, top-left 98.6, bottom-left **70.8**.
- the white paper in the bottom-left corner (~70) is darker than the printed ink in the bright top-right.
- the illumination estimate (the heavy-blur figure) shows the same brightness gradient running from the top-right to the bottom-left.
- **most usable mask:** the adaptive one.

**why does a single threshold struggle when the illumination changes?**
- a global threshold assumes one value separates ink from paper everywhere on the page.
- with uneven lighting, "paper" in a shadow can be darker than "ink" in a bright area, so no single value works.
  - a low T misses text in the bright areas (the ink there is lighter than T).
  - a high T marks the whole shadowed paper as ink.
- adaptive thresholding compares each pixel only with its own neighbourhood. the local lighting cancels out, so text is detected wherever it is darker than the paper around it.

### **task 2 - choosing the correct adaptive threshold strategy**
same image. `blockSize` = size of the local neighbourhood used to compute each pixel's threshold, `C` = constant subtracted from the local mean / gaussian-weighted mean.

In [ ]:
BLOCKS = [5, 11, 51]
CS = [0, 5, 15]
methods = {"mean": cv2.ADAPTIVE_THRESH_MEAN_C, "gaussian": cv2.ADAPTIVE_THRESH_GAUSSIAN_C}

results = {}
for mname, m in methods.items():
    imgs, titles = [], []
    for b in BLOCKS:
        for c in CS:
            mask = cv2.adaptiveThreshold(doc, 255, m, cv2.THRESH_BINARY_INV, b, c)
            results[(mname, b, c)] = mask
            imgs.append(mask)
            titles.append(f"{mname}  block={b}  C={c}")
    show(imgs, titles, cols=3, size=4, save=f"task2_{mname}_grid.png")

In [ ]:
# a simple numeric noise measure: number of small connected blobs (< 15 px) = speckle noise
def speckles(mask, max_area=15):
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask)
    return int((stats[1:, cv2.CC_STAT_AREA] < max_area).sum())

print(f"{'method':9s} {'block':>5s} {'C':>3s} {'ink %':>6s} {'speckles':>9s}")
for (mname, b, c), mask in results.items():
    print(f"{mname:9s} {b:5d} {c:3d} {100 * (mask > 0).mean():6.1f} {speckles(mask):9d}")

In [ ]:
best_mean = results[("mean", 51, 15)]
best_gauss = results[("gaussian", 51, 15)]
show([doc, best_mean, best_gauss], ["original", "best mean (51, C=15)", "best gaussian (51, C=15)"], size=5, save="task2_best.png")
cv2.imwrite("output/task2_best_mean.png", best_mean)

**answers** (from the two 3×3 grids and the table above; "speckles" = connected blobs smaller than 15 px)

1. **neighbourhood too small (blockSize 5):**
   - the window is smaller than a digit stroke, so inside thick strokes and plain paper the pixel is compared with near-identical neighbours.
   - with C = 0 the whole page becomes speckle noise (54-60% "ink", 900-1000 speckles).
   - with C = 15 the strokes break apart and mostly disappear (gaussian: 1.8% ink).
2. **neighbourhood too large (blockSize 51):**
   - the window starts to see the lighting gradient again, so it behaves a bit more like a global threshold.
   - strokes become thicker and bolder, small blobs merge (the "SUDOKU" label becomes a solid block) and dark folded edges show up as filled regions.
   - here it's still acceptable because 51 px is small compared with how slowly the lighting changes across the page.
3. **increasing C:**
   - C is subtracted from the local mean, so a pixel must be noticeably darker than its surroundings to count as ink.
   - larger C removes background noise (mean, block 51: 659 → 146 → 109 speckles for C = 0 → 5 → 15).
   - too large C with a small window erases faint or thin strokes (block 5, C 15).
4. **cleanest combination:** **blockSize 51, C = 15**.
   - fewest speckles (mean 109, gaussian 117).
   - solid, unbroken digits and grid lines.
   - blockSize 11 with C 5 is a close second: thinner strokes, but more noise (615-621 speckles).
5. **mean vs gaussian:** at the best settings they are almost identical, with mean slightly cleaner here (109 vs 117 speckles).
   - gaussian gives more weight to the centre pixel, so its threshold follows fine texture more closely.
   - that makes gaussian more sensitive to small blocks: at block 5, C 15 it erases more text than mean does.
   - for this image **mean** performs marginally better and is used as the final mask.

### **task 3 - quality-control system using otsu's thresholding**
image: `water_coins.jpg` from OpenCV's watershed tutorial (coins on a light background).

In [ ]:
coins = load("water_coins.jpg")
coins_gray = cv2.cvtColor(coins, cv2.COLOR_BGR2GRAY)

# THRESH_BINARY_INV because the coins are darker than the background
t_otsu, otsu_mask = cv2.threshold(coins_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
print("otsu threshold (original):", t_otsu)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].imshow(cv2.cvtColor(coins, cv2.COLOR_BGR2RGB)); axes[0].set_title("original"); axes[0].axis("off")
axes[1].hist(coins_gray.ravel(), 256, (0, 256), color="gray")
axes[1].axvline(t_otsu, color="red", ls="--", label=f"otsu T = {t_otsu:.0f}")
axes[1].set_title("grayscale histogram"); axes[1].legend()
axes[2].imshow(otsu_mask, cmap="gray"); axes[2].set_title("otsu binary mask"); axes[2].axis("off")
plt.tight_layout(); plt.savefig("output/task3_otsu.png", dpi=110); plt.show()
cv2.imwrite("output/task3_otsu_mask.png", otsu_mask)

#### **repeating the experiment with modified contrast / preprocessing**

In [ ]:
variants = {
    "original": coins_gray,
    "low contrast (x0.6 + 50)": cv2.convertScaleAbs(coins_gray, alpha=0.6, beta=50),
    "high contrast (x1.4 - 40)": cv2.convertScaleAbs(coins_gray, alpha=1.4, beta=-40),
    "gaussian blur 5x5": cv2.GaussianBlur(coins_gray, (5, 5), 0),
    "histogram equalized": cv2.equalizeHist(coins_gray),
}
imgs, titles = [], []
print(f"{'variant':28s} {'otsu T':>7s} {'foreground %':>13s}")
for name, img in variants.items():
    t, m = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    print(f"{name:28s} {t:7.0f} {100 * (m > 0).mean():13.1f}")
    imgs += [img, m]; titles += [name, f"mask (T = {t:.0f})"]
show(imgs, titles, cols=4, size=3.5, save="task3_variants.png")

**observation**
- **the histogram is bimodal:** a large bright peak (the white background) and a darker peak (the coins). otsu picked **T = 162**, in the valley between them. every coin comes out as a solid white disc.
- **changing contrast / preprocessing:**

| variant | otsu T | foreground |
|---|---|---|
| original | 162 | 55.9% |
| low contrast (×0.6 + 50) | 147 | 55.9% |
| high contrast (×1.4 − 40) | 167 | 55.2% |
| gaussian blur 5×5 | 162 | 56.5% |
| histogram equalized | 131 | 51.5% |

- **what the table shows:** the selected threshold moves with the contrast (147 → 167), but the segmented area stays almost identical (~55-56%). otsu followed the valley of the histogram automatically; a hand-picked threshold of 162 would have failed on the low-contrast version.
- **histogram equalization:** the one case that got worse. equalization boosts the coin engravings and the background texture, so the two peaks overlap more. the mask gets holes and ragged coin edges.
- **a limit of thresholding alone:** the coins touch, so the mask is effectively one connected blob and can't be used to count coins. that's what task 7 fixes.

**why is otsu useful when the correct threshold isn't known beforehand?**
- otsu tries every possible threshold and picks the one that maximises the variance between the two classes (equivalently, minimises the spread inside each class).
- the threshold is computed from each image's own histogram, so it adapts to changes in exposure and contrast with no manual tuning.
- the only assumption is that the histogram has two clear classes (bimodal).

### **task 4 - sorting objects by color**
image: `smarties.png` (the image-link sheet's choice for this task). target object: the **green** sweets.

In [ ]:
sweets = load("smarties.png")
hsv = cv2.cvtColor(sweets, cv2.COLOR_BGR2HSV)

# OpenCV hue range is 0-179, so pure green is ~60
restrictive = (np.array([55, 200, 200]), np.array([65, 255, 255]))
final = (np.array([40, 80, 60]), np.array([85, 255, 255]))

mask_r = cv2.inRange(hsv, *restrictive)
mask_f = cv2.inRange(hsv, *final)
extract = cv2.bitwise_and(sweets, sweets, mask=mask_f)

for name, m in [("restrictive", mask_r), ("final", mask_f)]:
    n = cv2.connectedComponentsWithStats(m)[0] - 1
    print(f"{name:11s} mask: {int((m > 0).sum()):6d} pixels, {n} blobs")

show([sweets, mask_r, mask_f, extract],
     ["original", "restrictive mask H55-65 S200+ V200+", "final mask H40-85 S80+ V60+", "extracted green objects"],
     cols=4, size=4, save="task4_hsv.png")
cv2.imwrite("output/task4_green_extracted.png", extract)

In [ ]:
# where do the green pixels actually sit in H, S and V? (explains why the restrictive range fails)
g = hsv[mask_f > 0]
for i, ch in enumerate("HSV"):
    print(f"{ch}: min {g[:, i].min():3d}  5%: {np.percentile(g[:, i], 5):5.0f}  median {np.median(g[:, i]):5.0f}  95%: {np.percentile(g[:, i], 95):5.0f}  max {g[:, i].max():3d}")

**observation**
- **target:** the **green** sweets. there is no yellow object in `smarties.png`.
- **restrictive mask** (H 55-65, S ≥ 200, V ≥ 200): only **877 pixels in 7 small fragments**, mostly slivers at the edges of the sweets. none of the sweets is captured as a whole.
- **final mask** (H 40-85, S ≥ 80, V ≥ 60): **5,362 pixels in 3 blobs**, which are the two whole green sweets plus the part of a third one cut off at the bottom-left edge. the extracted image contains only green objects.
- **the small black dots** inside the final mask are the specular highlights. they are nearly white (low saturation), so they fall outside any colour range.

**why the restrictive mask fails**
- the hue statistics of the actual green pixels show the problem: hue 64-83, median **67**, 95% of pixels at ≤ 75.
- the restrictive hue window (55-65) sits below most of the green, catching only the few pixels at the low end.
- requiring S ≥ 200 and V ≥ 200 also removes the shaded side of each sweet (lower V) and the edges (lower S).
- real objects are not one exact colour: shading, curvature and highlights spread their pixels over a range of H, S and V.
- the final range was widened to cover that spread. it stays well away from blue (~100-120) and red (0-10 / 170-179), so it doesn't pick up other sweets.

### **task 5 - detecting the boundary of a manufactured part**
image: `smarties.png` (objects with clear boundaries, plus soft shadows and highlights that create weak edges). the low threshold is kept fixed at 50 while the high threshold is increased.

In [ ]:
sweets_gray = cv2.cvtColor(sweets, cv2.COLOR_BGR2GRAY)
PAIRS = [(50, 100), (50, 200), (50, 350)]
edges = [cv2.Canny(sweets_gray, lo, hi) for lo, hi in PAIRS]

for (lo, hi), e in zip(PAIRS, edges):
    print(f"low={lo:3d} high={hi:3d}: {int((e > 0).sum()):6d} edge pixels")

show([sweets] + edges, ["original"] + [f"canny low={lo} high={hi}" for lo, hi in PAIRS], cols=4, size=4, save="task5_canny.png")
for (lo, hi), e in zip(PAIRS, edges):
    cv2.imwrite(f"output/task5_canny_{lo}_{hi}.png", e)

In [ ]:
# strong vs weak edges (hysteresis visualised for the middle pair):
#   strong = gradient above high, weak = between low and high. canny keeps weak pixels only if they connect to strong ones.
lo, hi = 50, 200
gx = cv2.Sobel(sweets_gray, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(sweets_gray, cv2.CV_64F, 0, 1, ksize=3)
mag = np.abs(gx) + np.abs(gy)          # L1 gradient, same as cv2.Canny's default
canny = cv2.Canny(sweets_gray, lo, hi)
vis = np.zeros_like(sweets)
vis[(canny > 0) & (mag >= hi)] = (255, 255, 255)   # strong edges -> white
vis[(canny > 0) & (mag < hi)] = (0, 200, 255)       # weak edges kept by hysteresis -> orange
show([sweets, vis], ["original", "white = strong, orange = weak (kept by hysteresis)"], size=5, save="task5_strong_weak.png")

**observation**
- **parameters:** the low threshold stays fixed at 50 while the high threshold rises from 100 to 200 to 350. the edge-pixel count drops from **3,726 to 3,196 to 3,032**.
- **strong edges:** the outer outline of every sweet against the white background. the gradient there is large, so it exceeds even the highest threshold, and it's present in all three results (white in the strong/weak figure).
- **weak edges:** the soft grey shadows next to the sweets and the rims of the bright highlights inside them (orange in the strong/weak figure). they only survive when they connect to a strong edge.
- **unwanted edges:**
  - the small closed loops inside the sweets (highlight reflections)
  - the arcs from shadows
  - at high = 100, extra streaks inside the red, orange and blue sweets from their colour shading
  - none of these belong to the boundary of the part.
- **missing edges:** on this clean image, the outer outlines stay complete even at high = 350. the risk shows up at the parts that are only partly in view (bottom-left corner): as the high threshold rises, less of each weak contour is started, so faint boundaries (like a low-contrast shadow side) would be the first to disappear.
- **best result:** high = 350, which keeps every boundary and removes most of the unwanted internal and shadow edges.

**how the two thresholds change the result (hysteresis)**
- pixels with a gradient above **high** are accepted immediately (strong edges).
- pixels between **low** and **high** are kept only if they connect to a strong edge; everything below low is discarded.
- **raising high** means fewer seed (strong) edges, so isolated texture and shadow edges disappear and the result gets cleaner. too high and real but weak boundaries are lost.
- **lowering low** lets edge chains continue further through weak sections, which closes gaps in the outlines but also lets noise attach to them.

### **task 6 - segmenting a region inside a medical image**
image: the input brain MRI (FLAIR) figure from the linked region-growing article, cropped to remove the figure border and caption. it shows a bright lesion on the left side of the image. seed A is placed inside the lesion (its brightest point after smoothing), seed B in normal tissue on the other side.

In [ ]:
fig_img = load("brain_mri.png", cv2.IMREAD_GRAYSCALE)
mri = fig_img[49:319, 31:337]           # crop out the figure frame and caption
mri = cv2.GaussianBlur(mri, (3, 3), 0)  # light smoothing so single noisy pixels don't block the growth
print("mri shape:", mri.shape)


def region_growing(image, seed, threshold):
    # region growing as in the manual: start at the seed, add 4-connected neighbours whose intensity
    # differs from the SEED intensity by less than the threshold.
    # (added: a 'visited' array so each pixel is tested only once - same result, much faster)
    mask = np.zeros_like(image, dtype=np.uint8)
    visited = np.zeros_like(image, dtype=bool)
    stack = [seed]
    seed_intensity = int(image[seed])
    while stack:
        x, y = stack.pop()
        if x < 0 or x >= image.shape[0] or y < 0 or y >= image.shape[1] or visited[x, y]:
            continue
        visited[x, y] = True
        if abs(int(image[x, y]) - seed_intensity) < threshold:
            mask[x, y] = 255
            stack.extend([(x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)])
    return mask


def overlay(gray, mask, seed):
    out = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    out[mask > 0] = (0.5 * out[mask > 0] + 0.5 * np.array([0, 0, 255])).astype(np.uint8)
    cv2.circle(out, (seed[1], seed[0]), 4, (0, 255, 0), -1)
    return out

In [ ]:
SEEDS = {"seed A (lesion)": (148, 122), "seed B (normal right-side tissue)": (125, 220)}   # (row, col)
THRESHOLDS = [20, 40, 60]

imgs, titles = [], []
print(f"{'seed':34s} {'seed value':>10s} {'thr':>4s} {'region px':>10s}")
for sname, seed in SEEDS.items():
    for t in THRESHOLDS:
        m = region_growing(mri, seed, t)
        print(f"{sname:34s} {int(mri[seed]):10d} {t:4d} {int((m > 0).sum()):10d}")
        imgs.append(overlay(mri, m, seed)); titles.append(f"{sname}\nthreshold = {t}")
        cv2.imwrite(f"output/task6_{sname.split()[1]}_thr{t}.png", m)
show([mri] + imgs[:3], ["mri (cropped)"] + titles[:3], cols=4, size=4, save="task6_seed_A.png")
show([mri] + imgs[3:], ["mri (cropped)"] + titles[3:], cols=4, size=4, save="task6_seed_B.png")

**observation**

| seed | seed value | threshold 20 | threshold 40 | threshold 60 |
|---|---|---|---|---|
| A: inside the lesion | 240 | 390 px | 1,472 px | 1,998 px |
| B: normal tissue, right side | 123 | 12,987 px | 21,980 px | 29,389 px |

- **seed A, the lesion:**
  - **threshold 20:** only a small patch around the seed. the lesion's own intensity varies, and those parts differ from 240 by more than 20.
  - **threshold 40:** covers most of the lesion.
  - **threshold 60:** fills almost the whole lesion and **stays inside it**, because the surrounding tissue (~150-160) is still more than 60 levels away from the seed. **60 gives the best lesion mask.**
- **seed B, normal tissue:**
  - **threshold 20:** already spreads over most of the grey/white matter of both hemispheres (the tissue intensities are similar and connected).
  - **threshold 60:** also takes in the bright ring of the skull/scalp. it never enters the lesion, which is far brighter than 123 + 60.
- **general effect of the threshold:** a larger threshold means a bigger region. too small and the region breaks up (the texture of the tissue stops it); too large and it leaks into neighbouring structures.

**why can changing the seed point significantly change the final segmented region?**
- region growing compares every pixel with the **seed's intensity** and only adds pixels **connected** to the region grown so far.
- the seed therefore decides two things: the reference intensity (240 vs 123 here) and where the growth starts.
- the same threshold gives a small lesion mask from seed A but half the brain from seed B.
- a seed on noise or on a boundary gives a misleading reference value, and a region of similar intensity that isn't connected to the seed is never reached.
- a single global threshold can't do this, which is why the task uses region growing to isolate one contiguous structure.

### **task 7 - separating touching coins (marker-based watershed)**
image: `water_coins.jpg`. the stages follow the manual's watershed pipeline.

In [ ]:
def watershed_coins(img, dist_ratio=0.5, verbose=True):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # 1. preprocessing: light blur to reduce noise before thresholding
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    # 2. thresholding (otsu, inverted: coins -> white)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    # 3. noise removal (morphological opening)
    kernel = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    # 4. sure background: dilate the objects - anything outside is certainly background
    sure_bg = cv2.dilate(opening, kernel, iterations=3)
    # 5. distance transform: each coin pixel -> distance to the nearest background pixel
    dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    # 6. sure foreground: only the pixels far from any edge (coin centres)
    _, sure_fg = cv2.threshold(dist, dist_ratio * dist.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)
    # 7. unknown region: between sure background and sure foreground (where the boundaries are)
    unknown = cv2.subtract(sure_bg, sure_fg)
    # 8. marker labelling: every sure-foreground blob gets its own label, background = 1, unknown = 0
    n_fg, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    # 9. watershed
    markers = cv2.watershed(img, markers)
    # 10. boundary visualisation
    result = img.copy()
    result[markers == -1] = (0, 0, 255)
    n_regions = len(np.unique(markers[markers > 1]))
    if verbose:
        print(f"distance ratio {dist_ratio}: {n_fg - 1} foreground markers, {n_regions} separated regions")
    return dict(thresh=thresh, sure_bg=sure_bg, dist=dist, sure_fg=sure_fg, unknown=unknown,
                markers=markers, result=result, n_markers=n_fg - 1, n_regions=n_regions)


r = watershed_coins(coins, 0.5)
n_naive = cv2.connectedComponents(r["thresh"])[0] - 1
print("connected regions in the plain threshold (naive count):", n_naive)

dist_vis = cv2.normalize(r["dist"], None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
show([coins, r["thresh"], r["sure_bg"], dist_vis, r["sure_fg"], r["unknown"], r["result"]],
     ["original", "threshold (otsu)", "sure background", "distance transform",
      "sure foreground (0.5 x max)", "unknown region", "final watershed"],
     cols=4, size=3.5, save="task7_pipeline.png")
cv2.imwrite("output/task7_watershed.png", r["result"])

In [ ]:
# colour each separated region so separation is easy to inspect
labels = r["markers"]
rng = np.random.default_rng(0)
palette = rng.integers(60, 255, size=(labels.max() + 1, 3), dtype=np.uint8)
colored = palette[np.clip(labels, 0, None)]
colored[labels == 1] = 0
colored[labels == -1] = 255
show([coins, colored], ["original", f"{r['n_regions']} separated regions (random colours)"], size=5, save="task7_regions.png")

**observation**
- **plain threshold:** the whole group of coins is **1 connected region** because they touch each other, so counting blobs would give 1 instead of 24.
- **how the pipeline fixes this:**
  - **sure background** (dilated mask): everything outside is certainly background.
  - **distance transform:** peaks at the centre of each coin and forms narrow "necks" where two coins touch.
  - **sure foreground** (0.5 × max distance): keeps only the coin centres, **24 separate blobs**. the necks fall below the threshold, so touching coins no longer connect.
  - **unknown region:** the rings between the centres and the sure background. this is where watershed decides the boundaries.
  - **watershed:** each marker floods outwards until it meets its neighbour, producing **24 separated regions**. that matches the 24 coins in the image.
- **visual check (coloured region image):**
  - every touching pair of coins is split, with the boundary running roughly along the contact line.
  - the boundaries are slightly irregular in a few places (e.g. a small bump at the contact between the coins at the upper right), because the red boundary follows the image gradient, not a perfect circle.

### **task 8 - tuning watershed for difficult object separation**
the task 7 pipeline with different distance-transform thresholds (as a fraction of the maximum distance).

In [ ]:
RATIOS = [0.3, 0.4, 0.42, 0.5, 0.7, 0.95]
runs = [watershed_coins(coins, d, verbose=False) for d in RATIOS]
show([x["result"] for x in runs], [f"ratio {d}: {x['n_markers']} markers, {x['n_regions']} regions" for d, x in zip(RATIOS, runs)],
     cols=3, size=4, save="task8_experiments.png")
show([x["sure_fg"] for x in runs], [f"sure foreground, ratio {d}" for d in RATIOS], cols=6, size=3, save="task8_sure_fg.png")

print(f"{'exp':>3s} {'distance threshold':>20s} {'foreground markers':>19s} {'regions':>8s}")
for i, (d, x) in enumerate(zip(RATIOS, runs), 1):
    print(f"{i:3d} {d:8.2f} x max = {d * x['dist'].max():5.1f} {x['n_markers']:19d} {x['n_regions']:8d}")
for d, x in zip(RATIOS, runs):
    cv2.imwrite(f"output/task8_ratio_{d}.png", x["result"])

**experiment table** (distance threshold as a fraction of the max distance, 24 px)

| experiment | distance threshold | foreground markers | regions | observed result |
|---|---|---|---|---|
| 1 | 0.30 × max (7.2 px) | 1 | 1 | nothing is separated. the sure-foreground blobs connect through the necks between touching coins, giving one marker for all coins |
| 2 | 0.40 × max (9.6 px) | 7 | 7 | partly separated: small groups of coins are still merged under one marker |
| 3 | 0.42 × max (10.1 px) | 21 | 21 | almost right. a few pairs of coins with a wide contact are still merged |
| 4 | 0.50 × max (12.0 px) | 24 | 24 | **every coin separated**, with clean boundaries along the contact lines |
| 5 | 0.70 × max (16.8 px) | 24 | 24 | still 24 coins. one small boundary artifact appears at a coin at the lower right |
| 6 | 0.95 × max (22.8 px) | 24 | 24 | still 24, but the markers are tiny, so the flooding is decided mostly by the image gradient. stray boundary lines appear **inside** several coins on the right |

**observation**
- **the neck:** where two coins touch, the distance transform has a "neck" whose value depends on how much the coins overlap. a sure-foreground threshold **below** the neck height merges the coins (experiments 1-3); **above** it separates them.
- **the count jumps sharply:** from 1 to 7 to 21 to 24 between 0.30 and 0.50, because most necks here are 7-12 px high.
- **why the top values still give 24:** all the coins are the same size and nearly circular, so the high thresholds never split one coin into two markers. the downside is quality: markers too small give the unknown region to watershed. the boundaries then depend on coin texture and become ragged (experiment 6).
- **most meaningful range:** about **0.45-0.6 × max**. every coin gets exactly one reasonably large marker, and the watershed lines follow the real contact lines.

### **task 9 - segmenting an image using k-means**
image: `dog.jpeg` (Packt OpenCV repo). workflow from the manual: reshape → float32 → `cv2.kmeans` → rebuild from the cluster centres.

In [ ]:
dog = load("dog.jpeg")
pixels = np.float32(dog.reshape((-1, 3)))                  # every pixel -> a (B, G, R) feature vector
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
n_orig = len(np.unique(dog.reshape(-1, 3), axis=0))

segs, titles = [], []
print(f"original: {n_orig} distinct colours\n")
for K in [2, 4, 6]:
    cv2.setRNGSeed(0)                                     # repeatable random initialisation
    compactness, labels, centers = cv2.kmeans(pixels, K, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    centers = np.uint8(centers)
    seg = centers[labels.flatten()].reshape(dog.shape)
    segs.append(seg); titles.append(f"K = {K}")
    cv2.imwrite(f"output/task9_kmeans_K{K}.png", seg)
    counts = np.bincount(labels.flatten(), minlength=K)
    order = np.argsort(-counts)
    err = np.sqrt(np.mean((dog.astype(float) - seg.astype(float)) ** 2))
    print(f"K={K}: {K} colours (reduced from {n_orig}), rms difference from original = {err:.1f}")
    for i in order:
        b, g, r_ = centers[i]
        print(f"    cluster RGB({r_:3d},{g:3d},{b:3d})  {100 * counts[i] / counts.sum():5.1f}% of pixels")

show([dog] + segs, ["original"] + titles, cols=4, size=4, save="task9_kmeans.png")

**observation**

| K | clusters | rms difference from original | major regions represented |
|---|---|---|---|
| 2 | 2 colours (from 28,661) | 22.7 | dark brown (78.9%): trees, shadow, most of the path. light pink-grey (21.1%): the dog, the woman, sunlit leaves and bright ground |
| 4 | 4 colours | 13.9 | darkest brown (50.9%): forest shade. mid brown (27.5%): path and trunks. light brown-pink (13.2%): sunlit ground and patches of leaves. near-white (8.4%): dog, shirt, sky gaps |
| 6 | 6 colours | 11.4 | the K = 4 groups split further. a separate salmon cluster (5.7%) picks out the pink trousers and warm sunlit path, and a grey-violet cluster (5.2%) separates the shirt from the white dog |

- **K = 2:** the strongest simplification. the image becomes a light-vs-dark mask, and the dog and woman are one silhouette together with random bright patches of leaves.
- **K = 4:** the scene splits into meaningful layers (shade, path, sunlit ground, bright subjects). the dog is clearly separated, but the woman's pink trousers are merged with the brown path.
- **K = 6:** closest to the original (lowest rms difference). it separates the clothing colours, while still reducing ~28,700 colours to 6.
- **choosing K:** the difference from the original drops quickly from K = 2 to K = 4, then more slowly (22.7 → 13.9 → 11.4). K = 4-6 is the useful range for this image.
- **the method groups by colour only:** clusters aren't connected regions. the same "sunlit" colour appears on the dog, the path and scattered leaves, because pixel position isn't part of the feature vector.

### **task 10 - design a segmentation system for an unknown image**
image: `smarties.png`. goal: segment every sweet from the white background (and ideally tell them apart). three approaches: **otsu thresholding**, **color-based (HSV) thresholding** and **k-means**.

In [ ]:
# method 1: otsu on grayscale (inverted: sweets are darker than the white background)
t10, m1 = cv2.threshold(sweets_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# method 2: HSV - background is white (very low saturation), every sweet is strongly coloured
S_MIN = 60
m2 = cv2.inRange(hsv, np.array([0, S_MIN, 0]), np.array([179, 255, 255]))
m2 = cv2.morphologyEx(m2, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

# method 3: k-means on colour with K = 6
cv2.setRNGSeed(0)
_, lab10, cen10 = cv2.kmeans(np.float32(sweets.reshape(-1, 3)), 6, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
m3 = np.uint8(cen10)[lab10.flatten()].reshape(sweets.shape)

def blobs(mask, min_area=150):
    n, _, st, _ = cv2.connectedComponentsWithStats(mask)
    return int((st[1:, cv2.CC_STAT_AREA] >= min_area).sum())

print(f"otsu threshold: {t10:.0f}")
print(f"method 1 (otsu):  {100 * (m1 > 0).mean():5.1f}% foreground, {blobs(m1)} objects")
print(f"method 2 (hsv):   {100 * (m2 > 0).mean():5.1f}% foreground, {blobs(m2)} objects")
counts10 = np.bincount(lab10.flatten(), minlength=6)
for i in np.argsort(-counts10):
    b, g, r_ = np.uint8(cen10)[i]
    print(f"method 3 cluster RGB({r_:3d},{g:3d},{b:3d}) {100 * counts10[i] / counts10.sum():5.1f}%")

show([sweets, m1, m2, m3], ["original", f"method 1: otsu (T = {t10:.0f})", f"method 2: HSV saturation > {S_MIN}", "method 3: k-means (K = 6)"],
     cols=4, size=4, save="task10_comparison.png")
for name, im in [("otsu", m1), ("hsv", m2), ("kmeans", m3)]:
    cv2.imwrite(f"output/task10_{name}.png", im)

**analysis of each method**

**method 1: otsu thresholding (grayscale), T = 176**
1. *assumption:* the histogram is bimodal. the objects are all darker than the background, and brightness alone separates them.
2. *identifies:* the white background vs the sweets. it finds 13 objects, covering 19.7% of the image.
3. *errors:*
   - holes inside the sweets, where the bright specular highlights are lighter than T
   - ragged edges on the lighter sweets (green, orange), because their brightness is close to the background
   - touching sweets merge into one blob
4. *most important parameter:* the threshold. it's chosen automatically, but the conversion to gray throws away colour, and that is the real limitation.

**method 2: colour-based thresholding (HSV saturation > 60) + 3×3 opening**
1. *assumption:* the background is unsaturated (white) and every object is strongly coloured, whatever its hue.
2. *identifies:* 13 solid objects covering 20.2%. every sweet, including the dark-brown ones, comes out as a filled disc with smooth edges.
3. *errors:*
   - touching sweets are still merged, since there's no separation step
   - a few small holes at the strongest highlights
   - slight shadow fringes
4. *most important parameter:* the saturation minimum. too low and the grey shadows join the objects; too high and the dark-brown sweets are lost first. their saturation is the lowest of all the sweets (5th percentile ≈ 106, median ≈ 197), so a minimum above ~100 starts cutting holes in them.

**method 3: k-means on BGR colour (K = 6)**
1. *assumption:* the image contains about K dominant colours, and objects of the same colour belong together.
2. *identifies:*
   - white background (79.5%)
   - red (8.7%)
   - blue (4.7%)
   - green (3.5%)
   - dark brown (2.0%)
   - it is the only method that also tells the sweets apart by colour.
3. *errors:*
   - the orange sweets have no cluster of their own and are merged into red
   - the sixth cluster (1.6%) is used for pinkish shadow and highlight pixels instead of a real object class
   - touching sweets of the same colour are still one region
4. *most important parameter:* K. with K = 6 the shadows take up one cluster, so orange doesn't get one; the result also depends on the random initialisation (fixed seed used).

**technical conclusion**
- for separating the sweets from the background, **HSV saturation thresholding gave the most meaningful regions**: solid, clean discs for every sweet, including the dark ones, with far fewer holes than otsu.
- **why it works for this image:** the background is almost pure white (saturation ≈ 0) while every object is strongly coloured, so saturation separates them much better than brightness, which varies with shading and highlights.
- otsu, working on brightness only, is confused by the highlights and by the light-coloured sweets.
- k-means is the better choice if the goal is to *classify* sweets by colour, but it wastes a cluster on shadows and merges orange with red.
- none of the three separates touching sweets. that would need watershed on top of the HSV mask.